# GL Form Parsing: ai_parse + Claude Combined Pipeline

Extract hierarchical key-value pairs from GL insurance forms using two
complementary paths, then merge for maximum coverage.

| Step | Tool | Purpose |
| --- | --- | --- |
| 1 | `ai_parse_document` | SQL-native layout analysis: tables, section hierarchy, AI figure descriptions |
| 2 | Claude Sonnet | One-shot PDF comprehension: best section granularity, highest KV count |
| 3 | Combined merge | Claude primary + ai_parse gap-fill + images \~\> **95%+ key coverage** |
| 4 | MLflow evaluation | Custom scorers: key coverage, value agreement, extraction count |

In [0]:
import json, re, base64, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from html.parser import HTMLParser

CATALOG, SCHEMA, VOLUME = "shm", "genai", "raw_pdfs"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

PDF_FILES = [
    f"{VOLUME_PATH}/Sample Contractors GL Jobsite Report Redacted.pdf",
    f"{VOLUME_PATH}/Sample Restaurant GL Report Redacted.pdf",
]

FORM_TYPE_MAP = {
    "Sample Contractors GL Jobsite Report Redacted.pdf": "GL_Contractors_Jobsite",
    "Sample Restaurant GL Report Redacted.pdf": "GL_Restaurant",
}

for p in PDF_FILES:
    print(p.split('/')[-1])

In [0]:
# ---------------------------------------------------------------------------
# ai_parse_document: layout analysis, KV extraction, figure descriptions
# ---------------------------------------------------------------------------

# Step 1: Parse PDFs
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.gl_parsed_docs")
spark.sql(f"""
    CREATE TABLE {CATALOG}.{SCHEMA}.gl_parsed_docs AS
    SELECT regexp_extract(path, '([^/]+)$', 1) AS filename,
           ai_parse_document(content, MAP('version', '2.0', 'descriptionElementTypes', '*')) AS parsed
    FROM read_files('{VOLUME_PATH}', format => 'binaryFile')
    WHERE path LIKE '%GL%Report%'
""")

# Step 2: Extract elements
elements = spark.sql(f"""
    WITH elems AS (
        SELECT filename, posexplode(
            from_json(to_json(parsed:document:elements),
                'ARRAY<STRUCT<content:STRING, type:STRING, page:INT, description:STRING>>')
        ) AS (idx, elem)
        FROM {CATALOG}.{SCHEMA}.gl_parsed_docs
    )
    SELECT filename, idx, elem.type AS element_type,
           elem.content AS content, elem.description AS description
    FROM elems ORDER BY filename, idx
""").collect()

# Step 3: Inline HTML table parser
class _TP(HTMLParser):
    def __init__(self):
        super().__init__()
        self.rows, self._row, self._cell, self._in = [], [], "", False
    def handle_starttag(self, tag, _):
        if tag in ('td','th'): self._in, self._cell = True, ""
    def handle_endtag(self, tag):
        if tag in ('td','th') and self._in:
            self._in = False; self._row.append(self._cell.strip())
        elif tag == 'tr' and self._row:
            self.rows.append(self._row); self._row = []
    def handle_data(self, d):
        if self._in: self._cell += d

def parse_table(html):
    p = _TP(); p.feed(html)
    return [(r[0], r[1] if len(r) >= 2 else None) for r in p.rows]

# Step 4: Extract KV pairs + figure descriptions
PAGE_MARKERS = {'page_header', 'page_footer', 'page_number'}
SKIP_LOGOS = {'nsr', 'national safety', 'ascot'}

ai_records, ai_images = [], []
for fname in sorted({r['filename'] for r in elements}):
    doc = [r for r in elements if r['filename'] == fname]
    section = subsection = None
    page_break = True

    for i, row in enumerate(doc):
        etype, content = row['element_type'], row['content'] or ''

        if etype in PAGE_MARKERS:
            page_break = True; continue

        if etype == 'figure':
            desc = (row['description'] or '').strip()
            ocr = content.strip()
            if any(p in (desc + ocr).lower() for p in SKIP_LOGOS): continue
            caption = ''
            if i+1 < len(doc) and doc[i+1]['element_type'] == 'caption':
                caption = (doc[i+1]['content'] or '').strip()
            description = desc or (f"{caption}: {ocr[:200]}" if ocr else caption)
            if description:
                ai_images.append(dict(
                    section='Images/Photographs', subsection=caption or 'Uncaptioned',
                    key=f"Image: {caption}" if caption else f"Figure #{len(ai_images)+1}",
                    value=description, filename=fname, tool='ai_parse'))
            continue

        if etype == 'section_header':
            if page_break or not section: section, subsection = content, None
            else: subsection = content
            page_break = False
        elif etype == 'table' and section:
            for k, v in parse_table(content):
                ai_records.append(dict(section=section, subsection=subsection,
                                       key=k, value=v, filename=fname, tool='ai_parse'))

ai_kv_all = pd.DataFrame(ai_records)
ai_images_df = pd.DataFrame(ai_images) if ai_images else pd.DataFrame(columns=ai_kv_all.columns)

print(f"ai_parse: {len(ai_kv_all)} KV pairs, {len(ai_images_df)} figure descriptions")
print(ai_kv_all.groupby('filename').size().to_string())
display(ai_kv_all.head(5))

In [0]:
# ---------------------------------------------------------------------------
# Claude Sonnet: one-shot PDF → hierarchical KV extraction
# ---------------------------------------------------------------------------
from openai import OpenAI

client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints",
)
MODEL = "databricks-claude-sonnet-4-5"

PROMPT = """Extract ALL key-value pairs from this GL insurance form as a JSON array.
Structure: [{"section": "...", "subsection": "..." or null, "key": "...", "value": "..."}]
Rules: Extract EVERY field from EVERY table/section. Use exact text from the form.
For checkboxes, use the selected option as the value. Return ONLY valid JSON."""

claude_records = []
for pdf_path in PDF_FILES:
    fname = pdf_path.split('/')[-1]
    with open(pdf_path, "rb") as f:
        pdf_b64 = base64.b64encode(f.read()).decode("utf-8")
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=32_000, temperature=0.0,
        messages=[{"role": "user", "content": [
            {"type": "document", "source": {
                "type": "base64", "media_type": "application/pdf", "data": pdf_b64}},
            {"type": "text", "text": PROMPT},
        ]}],
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith('```'): raw = raw.split('\n', 1)[1].rsplit('```', 1)[0]
    pairs = json.loads(raw)
    for kv in pairs:
        kv['filename'] = fname
        kv['tool'] = 'claude_sonnet'
    claude_records.extend(pairs)
    print(f"{fname}: {len(pairs)} KV pairs "
          f"({time.time()-t0:.0f}s, {resp.usage.prompt_tokens:,} in / {resp.usage.completion_tokens:,} out)")

claude_kv_all = pd.DataFrame(claude_records)
print(f"\nClaude total: {len(claude_kv_all)} KV pairs")
display(claude_kv_all.head(5))

In [0]:
# ---------------------------------------------------------------------------
# Combined pipeline: Claude primary + ai_parse gap-fill + figure descriptions
# ---------------------------------------------------------------------------

combined_records = []
for fname in sorted(set(ai_kv_all['filename'].unique()) | set(claude_kv_all['filename'].unique())):
    claude_doc = claude_kv_all[claude_kv_all['filename'] == fname]
    ai_doc = ai_kv_all[ai_kv_all['filename'] == fname]

    # Claude primary
    claude_keys = set(claude_doc['key'].str.lower().str.strip())
    for _, r in claude_doc.iterrows():
        combined_records.append(dict(section=r['section'], subsection=r.get('subsection'),
            key=r['key'], value=r['value'], source='claude', filename=fname, tool='combined'))

    # ai_parse gap-fill
    gap = [dict(section=r['section'], subsection=r.get('subsection'),
        key=r['key'], value=r['value'], source='ai_parse', filename=fname, tool='combined')
        for _, r in ai_doc.iterrows() if r['key'].lower().strip() not in claude_keys]
    combined_records.extend(gap)

# Add figure descriptions
for _, r in ai_images_df.iterrows():
    combined_records.append(dict(section=r['section'], subsection=r['subsection'],
        key=r['key'], value=r['value'], source='figures', filename=r['filename'], tool='combined'))

combined_kv = pd.DataFrame(combined_records)
TOOLS = {'ai_parse': ai_kv_all, 'claude_sonnet': claude_kv_all, 'combined': combined_kv}

# ── Summary metrics ──────────────────────────────────────────────────────
print("─── Extraction Summary ───")
for tool, df in TOOLS.items():
    for fname in sorted(df['filename'].unique()):
        d = df[df['filename'] == fname]
        short = fname.replace('Sample ', '').replace(' Redacted.pdf', '')
        print(f"  {tool:>15} | {short}: {len(d):>3} KV, {d['section'].nunique():>2} sections")

print("\n─── Pairwise Agreement ───")
from itertools import combinations
for fname in sorted(combined_kv['filename'].unique()):
    short = fname.replace('Sample ', '').replace(' Redacted.pdf', '')
    print(f"  {short}:")
    for t1, t2 in combinations(TOOLS, 2):
        d1 = TOOLS[t1][TOOLS[t1]['filename'] == fname]
        d2 = TOOLS[t2][TOOLS[t2]['filename'] == fname]
        m = d1.merge(d2, on='key', suffixes=('_1','_2'), how='inner').drop_duplicates('key')
        n = len(m)
        exact = int((m['value_1'].str.strip() == m['value_2'].str.strip()).sum()) if n else 0
        print(f"    {t1} vs {t2}: {n} shared, {exact/n*100:.0f}% exact" if n else
              f"    {t1} vs {t2}: 0 shared")

# ── Comparison chart ─────────────────────────────────────────────────────
colors = {'ai_parse': '#4c72b0', 'claude_sonnet': '#55a868', 'combined': '#888888'}
fnames = sorted(combined_kv['filename'].unique())
shorts = [f.replace('Sample ', '').replace(' Redacted.pdf', '') for f in fnames]
x = np.arange(len(fnames)); w = 0.25

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (title, fn) in zip(axes, [
    ('KV Pairs', lambda t, f: len(TOOLS[t][TOOLS[t]['filename']==f])),
    ('Sections', lambda t, f: TOOLS[t][TOOLS[t]['filename']==f]['section'].nunique()),
    ('Unique Keys', lambda t, f: len(set(TOOLS[t][TOOLS[t]['filename']==f]['key']) -
        set().union(*(set(TOOLS[o][TOOLS[o]['filename']==f]['key']) for o in TOOLS if o != t)))),
]):
    for i, t in enumerate(TOOLS):
        vals = [fn(t, f) for f in fnames]
        bars = ax.bar(x + (i-1)*w, vals, w, label=t, color=colors[t])
        for j, v in enumerate(vals): ax.text(x[j]+(i-1)*w, v+1, str(v), ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(shorts, fontsize=9)
    ax.set_title(title); ax.legend(fontsize=8)
plt.suptitle('ai_parse vs Claude vs Combined', fontweight='bold')
plt.tight_layout(); plt.show()

In [0]:
# ---------------------------------------------------------------------------
# MLflow Evaluation: key coverage, value agreement, extraction count
# ---------------------------------------------------------------------------
import mlflow
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback

EXPERIMENT_PATH = "/Users/scott.mckean@databricks.com/GL_Form_Parsing_Eval"
mlflow.set_experiment(EXPERIMENT_PATH)

# Reference = union of all tools' keys per form type
all_kv = pd.concat(TOOLS.values(), ignore_index=True)
ref_keys_by_form = {}
for fname in all_kv['filename'].unique():
    form_type = FORM_TYPE_MAP.get(fname, fname)
    ref_keys_by_form[form_type] = set(all_kv[all_kv['filename'] == fname]['key'].dropna())

# Consensus = majority vote
consensus = {}
for fname in all_kv['filename'].unique():
    for key in all_kv[all_kv['filename'] == fname]['key'].unique():
        vals = all_kv[(all_kv['filename'] == fname) & (all_kv['key'] == key)]['value'].dropna().str.strip().tolist()
        if vals: consensus[(fname, key)] = Counter(vals).most_common(1)[0][0]

# Build eval dataset
eval_data = []
for fname in sorted(all_kv['filename'].unique()):
    ft = FORM_TYPE_MAP.get(fname, fname)
    doc_cons = {k: v for (f, k), v in consensus.items() if f == fname}
    for tool_name, tool_df in TOOLS.items():
        extracted = {r['key']: str(r['value']).strip()
                     for _, r in tool_df[tool_df['filename'] == fname].iterrows()
                     if pd.notna(r['key'])}
        eval_data.append({
            "inputs": {"filename": fname, "tool": tool_name, "form_type": ft},
            "outputs": json.dumps(extracted),
            "expectations": {
                "reference_keys": json.dumps(list(ref_keys_by_form.get(ft, set()))),
                "consensus_values": json.dumps(doc_cons),
            },
        })
print(f"Eval: {len(eval_data)} rows ({len(TOOLS)} tools × {all_kv['filename'].nunique()} docs)")

@scorer
def key_coverage(*, outputs, expectations):
    ext = set(json.loads(outputs).keys())
    ref = set(json.loads(expectations['reference_keys']))
    return Feedback(name="key_coverage",
        value=round(len(ext & ref) / len(ref) * 100, 1) if ref else 0.0)

@scorer
def value_agreement(*, outputs, expectations):
    ext = json.loads(outputs)
    cons = json.loads(expectations['consensus_values'])
    shared = set(ext) & set(cons)
    matches = sum(1 for k in shared if ext[k].strip() == cons[k].strip())
    return Feedback(name="value_agreement",
        value=round(matches / len(shared) * 100, 1) if shared else 0.0)

@scorer
def extraction_count(*, outputs):
    return Feedback(name="extraction_count", value=len(json.loads(outputs)))

results = mlflow.genai.evaluate(
    data=eval_data,
    scorers=[key_coverage, value_agreement, extraction_count],
)
print(f"\n✅ Done | Run: {results.run_id}")
for k, v in sorted(results.metrics.items()): print(f"  {k}: {v}")
results_df = results.tables["eval_results"]
display(results_df)

In [0]:
# ---------------------------------------------------------------------------
# Evaluation Scorecard
# ---------------------------------------------------------------------------
scorecard = results_df[['extraction_count/value', 'key_coverage/value',
                        'value_agreement/value']].copy()
scorecard.columns = ['KV Pairs', 'Key Coverage %', 'Value Agreement %']
scorecard['tool'] = [d['inputs']['tool'] for d in eval_data]
scorecard['form_type'] = [d['inputs']['form_type'] for d in eval_data]

print("🎯 Scorecard")
display(scorecard.sort_values(['form_type', 'tool']).reset_index(drop=True))

avg = (scorecard.groupby('tool')[['KV Pairs', 'Key Coverage %', 'Value Agreement %']]
       .mean().round(1).sort_values('Key Coverage %', ascending=False))
print("\n📊 Averages Across Form Types")
display(avg)

# Chart
colors = {'ai_parse': '#4c72b0', 'claude_sonnet': '#55a868', 'combined': '#888888'}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, ['KV Pairs', 'Key Coverage %', 'Value Agreement %']):
    pivot = scorecard.pivot(index='form_type', columns='tool', values=metric)
    x = np.arange(len(pivot)); w = 0.25
    for i, tool in enumerate(sorted(pivot.columns)):
        vals = pivot[tool].values
        ax.bar(x + (i-1)*w, vals, w, label=tool, color=colors.get(tool, '#c44e52'))
        for j, v in enumerate(vals):
            ax.text(x[j]+(i-1)*w, v+max(vals)*0.02, f'{v:.0f}', ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels([f.replace('GL_','').replace('_','\n') for f in pivot.index], fontsize=9)
    ax.set_title(metric, fontweight='bold')
    if metric == 'KV Pairs': ax.legend(fontsize=8)
plt.suptitle('MLflow Evaluation by Form Type', fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
print(f"\n🔗 Experiment: {EXPERIMENT_PATH} | Run: {results.run_id}")